In [1]:
import sys,os
import numpy             as np
import matplotlib.pyplot as plt
import pandas            as pd
import seaborn           as sb

from itertools import product
from copy      import deepcopy
from time      import time
from tqdm      import tqdm

import warnings
warnings.filterwarnings('ignore')

from cobaya.run   import run

from source_code.likelihood import LSSlike

import matplotlib
from matplotlib import rc
rc('text', usetex=True)
rc('font', family='serif')
matplotlib.rcParams.update({'font.size': 18})

red    = '#8e001c'
yellow = '#ffb302'

sidelegend = {'bbox_to_anchor': (1.04,0.5), 
              'loc': "center left",
              'frameon': False}

camb_path = '/Users/chiaradeleo/myenv/lib/python3.12/site-packages'

In [2]:
H0_grid = np.linspace(66.5,67.5,10)

In [3]:


info = {'sampler': {'mcmc': {'max_tries':100000}},
                             #'covmat': 'LCDM_covmat_3x2pt.covmat'}},
        'likelihood': {'LSS': {'external': LSSlike,
                               'data_path': './mock_data/LCDM_test_galonly_IApoly',
                               'debug_mode': False,
                               'camb_path': camb_path,
                               'use_noiseless_cls': True}},
        'theory':{'camb': {'extra_args': {'dark_energy_model': 'ppf',
                                           'num_massive_neutrinos': 1,
                                           },
                           'path': camb_path,
                           'stop_at_error': False}}}


info['params'] = {'ombh2': 0.022445,
                  'omch2': 0.1205579307,
                  'ns': 0.96,
                  'As': 2.12605e-09,
                  'tau': 0.05,
                  'H0': {'latex': 'H_0',
                         'prior': {'max': 100.0,'min': 40.0},
                         'proposal': 0.5,
                         'ref': {'dist': 'norm','loc': 67.0,'scale': 1.0}},
                  'w': -1.,
                  'wa': 0.,
                  'mnu': 0.06,
                  'b0_poly': 0.830703,
                  'b1_poly': 1.190547,
                  'b2_poly': -0.928357,
                  'b3_poly': 0.423292} 


In [4]:
chi2 = []
for ind,H0 in enumerate(H0_grid):
    print('')
    print('Test point {}/{}'.format(ind+1,len(H0_grid)))
    info['sampler'] = {'evaluate': {'override': {'H0': H0}}}
    update_info,sampler = run(info)
    chi2.append(-2*sampler.logposterior.loglike)


Test point 1/10
[camb] `camb` module loaded successfully from /Users/chiaradeleo/myenv/lib/python3.12/site-packages/camb
Covmats inverted in 0.314
[model] *WARNING* Theories {camb.transfers, camb} do not appear to be actually used for anything
[evaluate] Initialized!
[evaluate] Looking for a reference point with non-zero prior.
[evaluate] Reference point:
   H0 = 66.5
[evaluate] Evaluating prior and likelihoods...


AttributeError: 

In [ ]:
plt.figure()
plt.plot(H0_grid,chi2,color=red,label='Standard',lw=3)
plt.axvline(x=info['params']['H0']['ref']['loc'],color='black',ls=':')
plt.xlabel(r'$H_0$')
plt.ylabel(r'$\chi^2(H_0)$')
plt.legend(**sidelegend);